In [1]:
import torch

if torch.cuda.is_available():
    print("GPU is available!")
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is NOT available. Please check your runtime settings.")

GPU is available!
Current CUDA device: 0
Device name: Tesla T4


# Video Segmentation and Object Tracking with SAM 3

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/segment-geospatial/blob/main/docs/examples/sam3_object_tracking.ipynb)

This notebook demonstrates how to use SAM 3 for video segmentation and object tracking.


## Installation

SAM 3 requires CUDA-capable GPU. Install with:


In [2]:
%pip install "segment-geospatial[samgeo3]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 632.9/632.9 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 117.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━

## Import Libraries


In [3]:
import os
from samgeo import SamGeo3Video, download_file

To use SamGeo 2, install it as:
	pip install segment-geospatial[samgeo2]


## Initialize Video Predictor

The `SamGeo3Video` class provides a simplified API for video segmentation. It automatically uses all available GPUs.


In [5]:
#verificar conexiona hugging face
sam = SamGeo3Video()

INFO 2025-12-07 06:02:30,318 178 sam3_video_predictor.py: 299: using the following GPU IDs: [0]
INFO 2025-12-07 06:02:30,320 178 sam3_video_predictor.py: 315: 


	*** START loading model on all ranks ***


INFO 2025-12-07 06:02:30,321 178 sam3_video_predictor.py: 317: loading model on rank=0 with world_size=1 -- this could take a while ...


Using GPUs: [0]


INFO 2025-12-07 06:02:40,716 178 sam3_video_base.py: 124: setting max_num_objects=10000 and num_obj_for_compile=16


GatedRepoError: 403 Client Error. (Request ID: Root=1-69351880-56a11baf31b8853b2225fef6;9f7b9997-a4da-4a3d-a8e4-546d675abf40)

Cannot access gated repo for url https://huggingface.co/facebook/sam3/resolve/main/config.json.
Your request to access model facebook/sam3 is awaiting a review from the repo authors.

This code snippet uses `torch.cuda.is_available()` to check for GPU presence. If a GPU is found, it will print details about it. You can place this cell after your imports to quickly verify the GPU status.

## Load a Video

You can load from different sources:
- MP4 video file
- Directory of JPEG frames
- Directory of GeoTIFFs (for remote sensing time series)


In [ ]:
url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/basketball.mp4"
video_path = download_file(url)

In [ ]:
sam.set_video(video_path)

In [ ]:
sam.show_video(video_path)

## Text-Prompted Segmentation

Use natural language to describe objects. SAM 3 finds all instances and tracks them.


In [ ]:
# Segment all players in the video
sam.generate_masks("player")

## Visualize Results

Customize player names:

In [ ]:
player_names = {}
for i in range(15):
    player_names[i] = f"Player {i}"
sam.show_frame(0, axis="on", show_ids=player_names)

![](https://github.com/user-attachments/assets/53c1752c-023a-4ae1-8e1a-6a83149220f6)

## Remove objects

In [ ]:
# Remove objects and re-propagate
sam.remove_object(obj_id=[5, 8, 12, 13])
sam.propagate()
sam.show_frame(0, show_ids=player_names)

![](https://github.com/user-attachments/assets/0b6566fa-a1ab-40c1-82cc-62212982d840)

## Save Results

Save masks as images or create an output video.


In [ ]:
os.makedirs("output", exist_ok=True)

# Save mask images
sam.save_masks("output/masks")

In [ ]:
# Save video with blended masks
sam.save_video("output/players_segmented.mp4", fps=60, show_ids=player_names)

In [ ]:
sam.show_video("output/players_segmented.mp4")

## Close Session

Close the session to free GPU resources.


In [ ]:
sam.close()

To completely shutdown and free all resources:

In [ ]:
sam.shutdown()